# 통계적 엄밀성 — 신뢰구간(CI) + Ablation Study

**목적:** TAI 저널 리뷰어는 두 가지를 요구합니다.

1. **신뢰구간 (Confidence Interval):** AUC 수치의 통계적 신뢰성 — 1000회 Bootstrap.
2. **Ablation Study:** 각 특징의 기여도 — "이 특징을 제거하면 성능이 얼마나 떨어지나?"

**기존 캐시 재사용** — blur_edge_results + improved_scan_results 파일로 즉시 실행.

**출력:**
- Table A: AUC ± 95% CI (bootstrap) — 논문 메인 테이블 업데이트용
- Table B: Ablation — 특징 제거 시 Δ Overall AUC
- Table C: SCAN marginal contribution (HF-Energy와의 상관도 포함)

In [ ]:
import os, subprocess, pickle
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy import stats

SEARCH_ROOTS=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..']
def _find(name,ftype='f',maxdepth=8):
    results=[]
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        try:
            out=subprocess.run(['find',root,'-maxdepth',str(maxdepth),'-type',ftype,'-name',name],
                               capture_output=True,text=True,timeout=15).stdout.strip()
            if out: results.extend([p for p in out.split('\n') if p])
        except: pass
    return sorted(set(results))

# blur_edge_results 캐시 (hf05, gl10, predl1 등)
# improved_scan_results 캐시 (sp_hf_scan_05 등)
all_fd = {}
for ds_name in ['CIFAR-10','TinyImageNet','ImageNet_eps8']:
    be_key = {'CIFAR-10':'features_blur_edge_CIFAR-10.pkl',
              'TinyImageNet':'features_blur_edge_TinyImageNet.pkl',
              'ImageNet_eps8':'features_blur_edge_ImageNet_eps8.pkl'}[ds_name]
    is_key = {'CIFAR-10':'feats_cifar10.pkl',
              'TinyImageNet':'feats_tinyimagenet.pkl',
              'ImageNet_eps8':'feats_imageneteps8.pkl'}[ds_name]

    be_cands=[p for p in _find(be_key) if 'blur_edge_results' in p]
    is_cands=[p for p in _find(is_key) if 'improved_scan_results' in p]

    if not be_cands or not is_cands:
        print(f"✗ {ds_name}: cache missing"); continue

    with open(be_cands[0],'rb') as f: be=pickle.load(f)
    with open(is_cands[0],'rb') as f: isf=pickle.load(f)
    merged=dict(be)
    for k in isf:
        if k not in ('labels','attacks'): merged[k]=isf[k]
    all_fd[ds_name]=merged
    print(f"✓ {ds_name}: {len(merged['labels'])} samples")

print("Caches loaded ✓")

In [ ]:
def zscore(arr):
    a=np.array(arr,dtype=float); return (a-a.mean())/(a.std()+1e-8)

def anomaly_score(fd, keys):
    """z-score 앙상블 anomaly = |mean_z - μ_clean|"""
    valid=[k for k in keys if k in fd]
    if not valid: return None
    z_stack=np.stack([zscore(fd[k]) for k in valid],axis=1)
    ens=z_stack.mean(axis=1)
    labels=np.array(fd['labels'])
    mu_c=ens[labels==0].mean()
    return np.abs(ens-mu_c)

def auc_with_ci(fd, keys, atk=None, n_boot=1000, ci=0.95, seed=42):
    """Bootstrap AUC ± CI 계산."""
    rng=np.random.default_rng(seed)
    anom=anomaly_score(fd,keys)
    if anom is None: return float('nan'),float('nan'),float('nan')
    labels=np.array(fd['labels']); attacks=np.array(fd['attacks'])

    if atk:
        mask=(attacks=='clean')|(attacks==atk); y,s=labels[mask],anom[mask]
    else:
        y,s=labels,anom

    if y.sum()==0 or (y==0).all() or (y==1).all():
        return float('nan'),float('nan'),float('nan')

    # 포인트 추정
    point_auc=roc_auc_score(y,s)

    # Bootstrap
    boot_aucs=[]
    n=len(y)
    for _ in range(n_boot):
        idx=rng.integers(0,n,n)
        y_b,s_b=y[idx],s[idx]
        if y_b.sum()==0 or (y_b==0).all() or (y_b==1).all(): continue
        try: boot_aucs.append(roc_auc_score(y_b,s_b))
        except: continue

    if len(boot_aucs)<10: return point_auc,float('nan'),float('nan')
    alpha=(1-ci)/2
    lo=np.percentile(boot_aucs,alpha*100)
    hi=np.percentile(boot_aucs,(1-alpha)*100)
    return point_auc, lo, hi

print("Bootstrap CI function ready (n_boot=1000, 95% CI) ✓")
print("Warning: will take ~2-5 min per dataset")

In [ ]:
RESULTS_DIR=Path('./ci_ablation_results'); RESULTS_DIR.mkdir(exist_ok=True)

# 최적 앙상블 키 (데이터셋별)
OPTIMAL_KEYS = {
    'CIFAR-10'      : ['hf_energy_s0p5'],   # HF-Energy 단일
    'TinyImageNet'  : ['sp_hf_scan_05','hf_energy_s0p5','predl1_jpeg'],
    'ImageNet_eps8' : ['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg'],
}

print("="*75)
print("  Table A: Main Results with 95% Bootstrap CI (n=1000)")
print("="*75)
print(f"  {'Dataset':<20} {'Attack':<8} {'AUC':>8} {'95% CI':>18}")
print(f"  {'-'*60}")

CI_CACHE=RESULTS_DIR/'main_ci.pkl'
if CI_CACHE.exists():
    with open(CI_CACHE,'rb') as f: ci_results=pickle.load(f)
    print("  (Loaded from cache)")
else:
    ci_results={}
    for ds_name,fd in all_fd.items():
        keys=OPTIMAL_KEYS.get(ds_name,['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg'])
        atks=sorted(set(a for a in fd['attacks'] if a!='clean'))
        ci_results[ds_name]={}
        for atk in atks+['overall']:
            atk_filter=None if atk=='overall' else atk
            pt,lo,hi=auc_with_ci(fd,keys,atk_filter)
            ci_results[ds_name][atk]=(pt,lo,hi)
    with open(CI_CACHE,'wb') as f: pickle.dump(ci_results,f)

for ds_name,atk_dict in ci_results.items():
    for atk,(pt,lo,hi) in atk_dict.items():
        if np.isnan(pt): continue
        ci_str=f"[{lo:.4f}, {hi:.4f}]" if not np.isnan(lo) else "N/A"
        print(f"  {ds_name:<20} {atk:<8} {pt:>8.4f}  {ci_str:>18}")
    print()
print("="*75)

In [ ]:
print()
print("="*75)
print("  Table B: Ablation Study — 특징 제거 시 Δ Overall AUC")
print("="*75)

ABLATION_CACHE=RESULTS_DIR/'ablation.pkl'
if ABLATION_CACHE.exists():
    with open(ABLATION_CACHE,'rb') as f: abl_results=pickle.load(f)
    print("  (Loaded from cache)")
else:
    abl_results={}
    for ds_name,fd in all_fd.items():
        full_keys=OPTIMAL_KEYS.get(ds_name,['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg'])
        full_auc,_,_=auc_with_ci(fd,full_keys,n_boot=200)
        abl_results[ds_name]={'full':full_auc,'ablations':{}}

        for remove_key in full_keys:
            ablated=[k for k in full_keys if k!=remove_key]
            if not ablated: continue
            abl_auc,lo,hi=auc_with_ci(fd,ablated,n_boot=200)
            abl_results[ds_name]['ablations'][remove_key]=(abl_auc,lo,hi,full_auc-abl_auc)

        # 각 특징 단독 AUC
        for k in full_keys:
            solo_auc,lo,hi=auc_with_ci(fd,[k],n_boot=200)
            abl_results[ds_name]['ablations'][f'solo_{k}']=(solo_auc,lo,hi,None)

    with open(ABLATION_CACHE,'wb') as f: pickle.dump(abl_results,f)

KEY_NAMES={'hf_energy_s0p5':'HF-Energy σ=0.5',
           'gauss_l1_s1p0':'GaussianL1 σ=1.0',
           'predl1_jpeg':'PredL1-JPEG',
           'sp_hf_scan_05':'Spatial-HF-SCAN σ=0.5'}

for ds_name,result in abl_results.items():
    full=result['full']
    print(f"\n  [{ds_name}]  Full ensemble AUC = {full:.4f}")
    print(f"  {'Feature removed':<28} {'Without AUC':>12} {'Δ AUC':>10} {'Impact'}")
    print(f"  {'-'*60}")
    for k,v in result['ablations'].items():
        if k.startswith('solo_'): continue
        abl_auc,lo,hi,delta=v
        name=KEY_NAMES.get(k,k)
        impact='★ 중요' if delta>0.02 else ('△ 보통' if delta>0.005 else '○ 미미')
        print(f"  Remove {name:<22} {abl_auc:>12.4f} {delta:>+10.4f}  {impact}")
    print(f"  {'─'*60}")
    print(f"  {'개별 특징 단독 AUC:':<28}")
    for k,v in result['ablations'].items():
        if not k.startswith('solo_'): continue
        orig_k=k[5:]
        solo_auc,lo,hi,_=v
        name=KEY_NAMES.get(orig_k,orig_k)
        print(f"  Solo {name:<24} {solo_auc:>12.4f}")
print("="*75)

In [ ]:
print()
print("="*70)
print("  Table C: SCAN 현저성 특징의 기여도 분석")
print("  (HF-Energy와의 상관도 + Marginal 기여)")
print("="*70)

SCAN_KEYS = ['noise_scan','jpeg_scan','median_scan']
SCAN_CACHE = RESULTS_DIR/'scan_marginal.pkl'
if SCAN_CACHE.exists():
    with open(SCAN_CACHE,'rb') as f: scan_results=pickle.load(f)
else:
    scan_results={}
    for ds_name,fd in all_fd.items():
        if not any(k in fd for k in SCAN_KEYS): continue
        hfe=np.array(fd.get('hf_energy_s0p5',[]))
        labels=np.array(fd['labels'])
        scan_results[ds_name]={}

        for sk in SCAN_KEYS:
            if sk not in fd: continue
            sv=np.array(fd[sk])
            # Spearman 상관도 with HF-Energy
            corr,pval=stats.spearmanr(hfe,sv)
            # 단독 AUC
            solo_auc,_,_=auc_with_ci(fd,[sk],n_boot=200)
            # 기존 앙상블 + SCAN 추가 시 marginal 기여
            base_keys=OPTIMAL_KEYS.get(ds_name,['hf_energy_s0p5'])
            base_auc,_,_=auc_with_ci(fd,base_keys,n_boot=200)
            with_scan_auc,_,_=auc_with_ci(fd,base_keys+[sk],n_boot=200)
            marginal=with_scan_auc-base_auc
            scan_results[ds_name][sk]={'solo':solo_auc,'corr':corr,'pval':pval,'marginal':marginal}

    with open(SCAN_CACHE,'wb') as f: pickle.dump(scan_results,f)

SCAN_NAMES={'noise_scan':'Noise-SCAN','jpeg_scan':'JPEG-SCAN','median_scan':'Median-SCAN'}
for ds_name,result in scan_results.items():
    print(f"\n  [{ds_name}]")
    print(f"  {'SCAN 특징':<18} {'단독 AUC':>10} {'vs HFE 상관도':>15} {'앙상블 기여 Δ':>15}")
    print(f"  {'-'*60}")
    for sk,v in result.items():
        name=SCAN_NAMES.get(sk,sk)
        corr_str=f"ρ={v['corr']:+.3f} (p={v['pval']:.3f})"
        impact='✗ 무효' if abs(v['marginal'])<0.002 else ('△ 미미' if v['marginal']<0.01 else '★ 유효')
        print(f"  {name:<18} {v['solo']:>10.4f} {corr_str:>15} {v['marginal']:>+13.4f}  {impact}")

print()
print("  해석: SCAN 스칼라 현저성은 HF-Energy와 낮은 상관도를 보이며")
print("  앙상블에 추가해도 marginal 기여가 거의 없음.")
print("  → 논문 Finding F2의 핵심 근거: '스칼라 요약이 공간 정보를 손실한다'")
print("="*70)

In [ ]:
print()
print("="*75)
print("  논문 최종 테이블 (AUC ± 95% CI) — 논문 Table 6 업데이트용")
print("="*75)
print(f"  {'Dataset':<20} {'FGSM':>16} {'PGD':>16} {'C&W':>16} {'Overall':>16}")
print(f"  {'-'*72}")

for ds_name,atk_dict in ci_results.items():
    row=f"  {ds_name:<20}"
    for atk in ['fgsm','pgd','cw','overall']:
        if atk not in atk_dict: row+=f"{'—':>16}"; continue
        pt,lo,hi=atk_dict[atk]
        if np.isnan(pt): row+=f"{'N/A':>16}"; continue
        if not np.isnan(lo):
            row+=f" {pt:.3f}±{(hi-lo)/2:.3f}"
        else:
            row+=f"{'':>5}{pt:.4f}{'':>5}"
    print(row)

print("="*75)
print()
print("  ± 값은 (95% CI 상한 - 하한) / 2 로 계산")
print("  Bootstrap n=1000, random seed=42 (재현 가능)")
print()
print("  코드 공개: 모든 실험 코드는 논문 제출 시 GitHub에 공개 예정")

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: AUC with error bars (bootstrap CI) ──────────────────────
atk_labels=['FGSM','PGD','C&W','Overall']
colors_ds=['steelblue','darkorange','green']
ds_names=list(ci_results.keys())

ax=axes[0]
x=np.arange(len(atk_labels)); w=0.25
for i,(ds_name,c) in enumerate(zip(ds_names,colors_ds)):
    pts=[]; los=[]; his=[]
    for atk in ['fgsm','pgd','cw','overall']:
        if atk in ci_results[ds_name]:
            pt,lo,hi=ci_results[ds_name][atk]
            pts.append(pt if not np.isnan(pt) else 0)
            los.append(pt-lo if not np.isnan(lo) else 0)
            his.append(hi-pt if not np.isnan(hi) else 0)
        else:
            pts.append(0); los.append(0); his.append(0)
    ax.bar(x+i*w, pts, w, label=ds_name, color=c, alpha=0.8, edgecolor='black')
    ax.errorbar(x+i*w, pts, yerr=[los,his], fmt='none', color='black', capsize=4, lw=1.5)

ax.set_xticks(x+w); ax.set_xticklabels(atk_labels)
ax.set_ylabel('AUC'); ax.set_ylim(0,1.15)
ax.set_title('Main Results with 95% CI\n(Bootstrap, n=1000)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
ax.axhline(0.5, color='red', lw=1, linestyle='--')

# ── Plot 2: Ablation — feature removal impact ──────────────────────
ax=axes[1]
all_abl_data={}
for ds_name,result in abl_results.items():
    for k,(abl_auc,lo,hi,delta) in result['ablations'].items():
        if k.startswith('solo_'): continue
        name=KEY_NAMES.get(k,k)
        if name not in all_abl_data: all_abl_data[name]={}
        all_abl_data[name][ds_name]=delta

feat_names_abl=list(all_abl_data.keys())
ds_names_abl=list({d for v in all_abl_data.values() for d in v})
x2=np.arange(len(feat_names_abl)); w2=0.25
for i,(ds,c) in enumerate(zip(ds_names_abl,colors_ds)):
    deltas=[all_abl_data[f].get(ds,0) for f in feat_names_abl]
    ax.bar(x2+i*w2, deltas, w2, label=ds, color=c, alpha=0.8, edgecolor='black')

ax.set_xticks(x2+w2); ax.set_xticklabels([f[:12] for f in feat_names_abl],
                                           rotation=15,fontsize=8)
ax.set_ylabel('Δ Overall AUC (when removed)')
ax.set_title('Ablation: Impact of Removing Each Feature', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3,axis='y')
ax.axhline(0, color='black', lw=0.5)

# ── Plot 3: SCAN marginal contribution ─────────────────────────────
ax=axes[2]
if scan_results:
    ds_scan=list(scan_results.keys())
    scan_feat_names=['Noise-SCAN','JPEG-SCAN','Median-SCAN']
    x3=np.arange(len(scan_feat_names)); w3=0.25
    for i,(ds,c) in enumerate(zip(ds_scan,colors_ds)):
        marginals=[scan_results[ds].get(sk,{}).get('marginal',0)
                   for sk in ['noise_scan','jpeg_scan','median_scan']]
        ax.bar(x3+i*w3,marginals,w3,label=ds,color=c,alpha=0.8,edgecolor='black')

    ax.set_xticks(x3+w3); ax.set_xticklabels(scan_feat_names)
    ax.set_ylabel('Marginal Δ AUC (added to ensemble)')
    ax.set_title('SCAN Marginal Contribution\n(+ to best ensemble)', fontsize=11)
    ax.legend(fontsize=9); ax.grid(alpha=0.3,axis='y')
    ax.axhline(0,color='red',lw=1,linestyle='--',label='Zero contribution')

plt.tight_layout()
fig_path=RESULTS_DIR/'ci_ablation_summary.png'
plt.savefig(fig_path,dpi=120,bbox_inches='tight')
plt.show()
print(f"Figure saved → {fig_path}")